# E-Commerce Analytics System
# Notebook 5: Advanced SQL Analytics

This notebook demonstrates advanced SQL concepts required in the assignment:

- Window Functions
- Common Table Expressions (CTEs)
- LAG / LEAD
- DENSE_RANK
- Running Totals
- Moving Averages
- NTILE Segmentation
- Year-over-Year Analysis
- First / Last Purchase Category
- Cumulative Revenue Distribution
- Cohort & Retention Analysis


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(Path("database")/"ecommerce.db")

def run_query(title, query):
    print("="*90)
    print(title)
    df = pd.read_sql(query, conn)
    display(df.head(30))
    return df


## Rank Customers by Lifetime Value

In [ ]:
run_query("""Rank Customers by Lifetime Value""", """
SELECT
c.customer_id,
c.customer_name,
ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) AS lifetime_value,
DENSE_RANK() OVER(
ORDER BY SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) DESC
) AS customer_rank
FROM customers c
JOIN orders o ON c.customer_id=o.customer_id
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY c.customer_id,c.customer_name;
""")

## Running Revenue by Region

In [ ]:
run_query("""Running Revenue by Region""", """
WITH daily AS(
SELECT
o.region_code,
date(o.order_date) order_day,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) revenue
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY o.region_code,date(o.order_date)
)
SELECT
region_code,
order_day,
ROUND(revenue,2) daily_revenue,
ROUND(
SUM(revenue) OVER(
PARTITION BY region_code
ORDER BY order_day
),2) running_total
FROM daily;
""")

## 7-Day Moving Average Revenue

In [ ]:
run_query("""7-Day Moving Average Revenue""", """
WITH daily AS(
SELECT
date(order_date) order_day,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) revenue
FROM orders o
JOIN order_items oi
ON o.order_id=oi.order_id
GROUP BY date(order_date)
)
SELECT
order_day,
ROUND(revenue,2) revenue,
ROUND(
AVG(revenue) OVER(
ORDER BY order_day
ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
),2) moving_avg
FROM daily;
""")

## Days Between Consecutive Orders

In [ ]:
run_query("""Days Between Consecutive Orders""", """
SELECT
customer_id,
order_date,
LAG(order_date) OVER(
PARTITION BY customer_id
ORDER BY order_date
) previous_order,
JULIANDAY(order_date)-
JULIANDAY(
LAG(order_date) OVER(
PARTITION BY customer_id
ORDER BY order_date)
) days_gap
FROM orders;
""")

## Monthly Revenue Growth (CTE)

In [ ]:
run_query("""Monthly Revenue Growth (CTE)""", """
WITH monthly AS(
SELECT
strftime('%Y-%m',order_date) month,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) revenue
FROM orders o
JOIN order_items oi
ON o.order_id=oi.order_id
GROUP BY month
)
SELECT
month,
ROUND(revenue,2) revenue,
ROUND(
100.0*(revenue-LAG(revenue) OVER(ORDER BY month))
/
LAG(revenue) OVER(ORDER BY month),2
) growth_percent
FROM monthly;
""")

## Customer Quartiles using NTILE

In [ ]:
run_query("""Customer Quartiles using NTILE""", """
WITH spending AS(
SELECT
c.customer_id,
ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) total_value
FROM customers c
JOIN orders o ON c.customer_id=o.customer_id
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY c.customer_id
)
SELECT
customer_id,
total_value,
NTILE(4) OVER(ORDER BY total_value DESC) quartile
FROM spending;
""")

## First vs Last Purchased Category

In [ ]:
run_query("""First vs Last Purchased Category""", """
WITH purchases AS(
SELECT
o.customer_id,
o.order_date,
p.category
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
JOIN products p ON oi.product_id=p.product_id
)
SELECT
customer_id,
FIRST_VALUE(category) OVER(
PARTITION BY customer_id
ORDER BY order_date
) first_category,
LAST_VALUE(category) OVER(
PARTITION BY customer_id
ORDER BY order_date
ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
) last_category
FROM purchases;
""")

## Top Revenue Contributors

In [ ]:
run_query("""Top Revenue Contributors""", """
WITH revenue AS(
SELECT
c.customer_id,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) revenue
FROM customers c
JOIN orders o ON c.customer_id=o.customer_id
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY c.customer_id
)
SELECT
customer_id,
ROUND(revenue,2) revenue,
ROUND(
100.0*
SUM(revenue) OVER(ORDER BY revenue DESC)
/
SUM(revenue) OVER(),2
) cumulative_percent
FROM revenue
ORDER BY revenue DESC;
""")

## Customer Cohort Retention

In [ ]:
run_query("""Customer Cohort Retention""", """
WITH first_purchase AS(
SELECT
customer_id,
MIN(strftime('%Y-%m',order_date)) cohort_month
FROM orders
GROUP BY customer_id
),
activity AS(
SELECT
o.customer_id,
f.cohort_month,
strftime('%Y-%m',o.order_date) activity_month
FROM orders o
JOIN first_purchase f
ON o.customer_id=f.customer_id
)
SELECT
cohort_month,
activity_month,
COUNT(DISTINCT customer_id) active_customers
FROM activity
GROUP BY cohort_month,activity_month
ORDER BY cohort_month,activity_month;
""")

## Close Database

In [ ]:
conn.close()
print("Advanced SQL analysis completed.")